# **Pre-Trained Model — EfficientNetV2B1**

EfficientNetV2B1 was selected as an upgraded architecture to push performance beyond EfficientNetB0.

### **Why EfficientNetV2?**

EfficientNetV2 is a redesigned version of the original EfficientNet family. Key improvements over V1:
- **Fused-MBConv blocks** in early layers replace depthwise separable convolutions, offering faster training and better accuracy on smaller datasets
- **Progressive learning** during pretraining (smaller images early, larger later) produces more robust feature representations
- **Better training efficiency** — converges in significantly fewer epochs than V1 for the same accuracy level

### **Why B1 specifically?**

B1 is the second-smallest V2 variant (~8M parameters vs B0's 5M). It fits within a 6GB VRAM budget at batch_size=16 while providing meaningful capacity gains over B0. Its native input resolution is **240x240**, which captures more dermoscopic detail than the 224x224 used by B0 — important for fine-grained texture tasks like HAM10000.

- **Suitability for dermoscopy**: The Fused-MBConv blocks in early layers are better at capturing low-level texture patterns (pigment networks, vascular structures) compared to the depthwise separable convolutions in V1
- **Transfer learning rationale**: V2 models were pretrained with a stronger augmentation recipe and progressive resizing, giving richer initializations for medical image fine-tuning

### Imports

In [ ]:
import tensorflow as tf
# Prevent TF from grabbing all GPU memory at once
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

In [ ]:
# NOTE: Using float32 for stable fine-tuning — mixed_float16 causes
# gradient overflow when unfreezing backbone layers
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('float32')
print('Policy:', mixed_precision.global_policy().name)

In [ ]:
import os
import sys
import keras
import keras.applications
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.preprocessing import label_binarize

if os.getcwd().endswith('models'):
    os.chdir('..')

from utils.utils_model import *
from utils.utils_augmentation import *
from utils.utils_preproc import *

In [ ]:
gpus = tf.config.list_physical_devices('GPU')
print("GPUs available:", gpus)
print("TF built with CUDA:", tf.test.is_built_with_cuda())
print("GPU available:", len(gpus) > 0)

### **Data** Configuration

In [ ]:
with open("label2idx.json", "r") as f:
    label2idx = json.load(f)

N_CLASSES  = len(label2idx)
BATCH_SIZE = 16        # Reduced from 32 to fit V2B1 in 6GB VRAM
IMG_SIZE   = 240       # Native resolution for EfficientNetV2B1

train_df = pd.read_csv('data/augmented_metadata.csv')
val_df   = pd.read_csv('data/val_split.csv')
test_df  = pd.read_csv('data/test_split.csv')

# Normalise column names and encode labels for all splits
for df in [train_df, val_df, test_df]:
    if 'cleaned_path' in df.columns and 'image_path' in df.columns:
        df.drop(columns=['image_path'], inplace=True)
    df.rename(columns={'cleaned_path': 'image_path'}, inplace=True)
    df['dx_encoded'] = df['dx'].map(label2idx).astype(int)

# Build datasets — EfficientNetV2 handles normalization internally,
# no external preprocess_fn needed (same as V1)
train_ds = make_dataset(train_df, output_shape=(IMG_SIZE, IMG_SIZE),
                        shuffle=True, repeat=True, batch_size=BATCH_SIZE)
val_ds   = make_dataset(val_df,   output_shape=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE)
test_ds  = make_dataset(test_df,  output_shape=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE)

STEPS_PER_EPOCH    = len(train_df) // BATCH_SIZE
class_weights_dict = make_class_weights(train_df)

In [ ]:
print(f"Steps per epoch: {STEPS_PER_EPOCH}")
print(f"Class weights: {class_weights_dict}")

# Sanity check — verify pixel range (should be 0-255, V2 normalizes internally)
batch = next(iter(train_ds))
imgs, _ = batch
print(f"Image dtype: {imgs.dtype}")
print(f"Image min/max: {imgs.numpy().min():.1f} / {imgs.numpy().max():.1f}  (expected: 0 / 255)")

### **Model** Configuration

In [ ]:
def build_efficientnetv2b1():
    base = keras.applications.EfficientNetV2B1(
        include_top=False,
        weights="imagenet",
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        pooling=None
    )
    base.trainable = False

    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base(inputs, training=False)

    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dropout(0.5)(x)
    x = keras.layers.Dense(256, activation="relu",
                           kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Dropout(0.4)(x)
    outputs = keras.layers.Dense(N_CLASSES)(x)   # logits — no softmax

    return keras.Model(inputs, outputs, name="efficientnetv2b1")

### Phase 1 — Train head only (backbone frozen)

In [ ]:
model_v2b1 = build_efficientnetv2b1()
model_v2b1.summary()

model_v2b1.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy", BalancedAccuracy(N_CLASSES)]
)

callbacks_p1 = get_callbacks(
    checkpoint_path="checkpoints/model_V2B1_phase1.weights.h5",
    model=model_v2b1,
    max_diff=0.20,
    patience_es=10,
    patience_lr=5
)

history_p1 = model_v2b1.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    steps_per_epoch=STEPS_PER_EPOCH,
    class_weight=class_weights_dict,
    callbacks=callbacks_p1
)

plot_history(history_p1, "EfficientNetV2B1 — Phase 1 (head only)")

### Phase 2 — Fine-tune full backbone

In [ ]:
# Rebuild and reload Phase 1 weights
model_v2b1_p2 = build_efficientnetv2b1()
model_v2b1_p2.load_weights("checkpoints/model_V2B1_phase1.weights.h5")
print("Phase 1 weights loaded successfully!")

# Verify starting point is healthy
model_v2b1_p2.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy", BalancedAccuracy(N_CLASSES)]
)
val_loss, val_acc, val_bal = model_v2b1_p2.evaluate(val_ds)
print(f"Val loss after loading Phase 1 weights: {val_loss:.4f}")
print(f"Val accuracy: {val_acc:.4f}  (should match Phase 1 best)")

In [ ]:
# Unfreeze full backbone, keep BN layers frozen
base = model_v2b1_p2.get_layer('efficientnetv2-b1')
base.trainable = True

for layer in base.layers:
    if isinstance(layer, keras.layers.BatchNormalization):
        layer.trainable = False

trainable_params = sum([tf.size(w).numpy() for w in model_v2b1_p2.trainable_weights])
print(f"Trainable params: {trainable_params:,}")

model_v2b1_p2.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy", BalancedAccuracy(N_CLASSES)]
)

callbacks_p2 = get_callbacks(
    checkpoint_path="checkpoints/model_V2B1_phase2.weights.h5",
    model=model_v2b1_p2,
    max_diff=0.20,
    patience_es=8,
    patience_lr=4
)

history_p2 = model_v2b1_p2.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    steps_per_epoch=STEPS_PER_EPOCH,
    class_weight=class_weights_dict,
    callbacks=callbacks_p2
)

plot_history(history_p2, "EfficientNetV2B1 — Phase 2 (full fine-tune)")

### Model Evaluation

In [ ]:
# Load best checkpoint before evaluating
model_v2b1_p2.load_weights("checkpoints/model_V2B1_phase2.weights.h5")
print("Best Phase 2 weights loaded")

results_v2b1 = evaluate_model(model_v2b1_p2, test_ds, label2idx, "EfficientNetV2B1")